# Module 5 | Lab: Customer Segmentation Pipeline
### Mall Customers — K-Means & Hierarchical Clustering

**Objective:** Segment mall customers into distinct groups, interpret the segments, and provide actionable business recommendations.


## Setup
The dataset is loaded directly from a URL below — no manual upload needed. Just run the cells in order.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.metrics import silhouette_score
from scipy.cluster.hierarchy import dendrogram, linkage

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (8, 5)


---
## Task 1 — Load and Explore the Data


In [ ]:
DATA_URL = "https://raw.githubusercontent.com/gakudo-ai/open-datasets/refs/heads/main/Mall_Customers.csv"
df = pd.read_csv(DATA_URL)

print("Shape:", df.shape)
df.head()


In [ ]:
df.info()


In [ ]:
df.describe()


**Note:** the two main clustering features are **Annual Income (k$)** and **Spending Score (1-100)**.
`Age` and `Gender` are supporting features we'll use later to enrich the interpretation.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
sns.histplot(df["Age"], kde=True, ax=axes[0], color="steelblue")
axes[0].set_title("Age distribution")

sns.histplot(df["Annual Income (k$)"], kde=True, ax=axes[1], color="seagreen")
axes[1].set_title("Annual Income distribution")

sns.histplot(df["Spending Score (1-100)"], kde=True, ax=axes[2], color="indianred")
axes[2].set_title("Spending Score distribution")

plt.tight_layout()
plt.show()


In [ ]:
sns.pairplot(df.drop(columns=["CustomerID"]), hue="Gender", corner=True)
plt.show()


**Do you already see natural groupings?**
Look closely at the *Annual Income vs Spending Score* panel in the pairplot above — you should be able to spot roughly five visually separated "blobs" of points. That's a strong hint about how many clusters (K) we should expect to find in Task 2.

---
## Task 2 — K-Means with Elbow and Silhouette


In [ ]:
features = df[["Annual Income (k$)", "Spending Score (1-100)"]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)
X_scaled[:5]


In [ ]:
wcss = []          # inertia_ for each K (elbow method)
sil_scores = []     # silhouette score for each K
K_range = range(2, 11)

for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_scaled)
    wcss.append(km.inertia_)
    sil_scores.append(silhouette_score(X_scaled, labels))

results = pd.DataFrame({"K": list(K_range), "WCSS": wcss, "Silhouette": sil_scores})
results


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].plot(K_range, wcss, marker="o", color="steelblue")
axes[0].set_xlabel("Number of clusters (K)")
axes[0].set_ylabel("WCSS (inertia)")
axes[0].set_title("Elbow Method")

axes[1].plot(K_range, sil_scores, marker="o", color="indianred")
axes[1].set_xlabel("Number of clusters (K)")
axes[1].set_ylabel("Silhouette Score")
axes[1].set_title("Silhouette Score by K")

plt.tight_layout()
plt.show()


**Choosing the final K:**
- The elbow chart should show WCSS dropping sharply up to around K=5, then flattening out — the "elbow" sits near K=5.
- The silhouette chart typically peaks at K=5 (or is very close to the peak), confirming that 5 clusters are well separated and internally compact.

*Fill in after running the cells above:* State the K you are choosing and justify it using the two numbers you actually observed (e.g. "Elbow flattens at K=5; silhouette score peaks at 0.XX at K=5, so I selected K=5").

In [ ]:
FINAL_K = 5  # <-- update this if your own results point to a different K

kmeans = KMeans(n_clusters=FINAL_K, random_state=42, n_init=10)
df["KMeans_Cluster"] = kmeans.fit_predict(X_scaled)
df.head()


In [ ]:
plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
    hue="KMeans_Cluster", palette="tab10", s=70
)
plt.title(f"K-Means Clusters (K={FINAL_K})")
plt.legend(title="Cluster")
plt.show()


---
## Task 3 — Hierarchical Clustering


In [ ]:
linked_ward = linkage(X_scaled, method="ward")

plt.figure(figsize=(12, 6))
dendrogram(linked_ward, truncate_mode="lastp", p=30)
plt.title("Dendrogram (Ward linkage)")
plt.xlabel("Cluster size (truncated)")
plt.ylabel("Distance")
plt.show()


Cut the dendrogram at a height that gives the same number of clusters as your K-Means result, and compare the two groupings.

In [ ]:
hier_ward = AgglomerativeClustering(n_clusters=FINAL_K, linkage="ward")
df["Hierarchical_Ward_Cluster"] = hier_ward.fit_predict(X_scaled)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
    hue="Hierarchical_Ward_Cluster", palette="tab10", s=70
)
plt.title(f"Hierarchical Clustering — Ward linkage (K={FINAL_K})")
plt.legend(title="Cluster")
plt.show()


In [ ]:
# Compare K-Means vs Hierarchical (Ward) groupings
pd.crosstab(df["KMeans_Cluster"], df["Hierarchical_Ward_Cluster"])


If the two methods agree, most of the mass in the cross-tab above will sit on a near-diagonal pattern (each K-Means cluster maps cleanly onto one hierarchical cluster).

In [ ]:
# Try complete linkage and compare balance
linked_complete = linkage(X_scaled, method="complete")

hier_complete = AgglomerativeClustering(n_clusters=FINAL_K, linkage="complete")
df["Hierarchical_Complete_Cluster"] = hier_complete.fit_predict(X_scaled)

print("Ward linkage cluster sizes:\n", df["Hierarchical_Ward_Cluster"].value_counts().sort_index())
print("\nComplete linkage cluster sizes:\n", df["Hierarchical_Complete_Cluster"].value_counts().sort_index())


**Which linkage gives more balanced clusters?** Compare the two size distributions printed above — the one with sizes closer to each other (less skewed) is the more balanced result. Ward linkage usually produces more evenly sized, compact clusters; complete linkage can isolate small outlier groups.

---
## Task 4 — Interpret the Clusters


In [ ]:
cluster_profile = df.groupby("KMeans_Cluster")[["Age", "Annual Income (k$)", "Spending Score (1-100)"]].mean().round(1)
cluster_profile["Count"] = df["KMeans_Cluster"].value_counts().sort_index()
cluster_profile


**Name each segment** (replace the placeholders below once you've looked at the table above — typical Mall Customers segments look like this, but confirm against YOUR numbers):

| Cluster | Typical profile | Suggested name |
|---|---|---|
| 0 | Low income, low spending | Budget-conscious / cautious |
| 1 | Low income, high spending | Impulsive spenders |
| 2 | High income, low spending | Cautious savers |
| 3 | High income, high spending | Premium / target customers |
| 4 | Mid income, mid spending | Average / standard customers |


In [ ]:
plt.figure(figsize=(9, 7))
sns.scatterplot(
    data=df, x="Annual Income (k$)", y="Spending Score (1-100)",
    hue="KMeans_Cluster", palette="tab10", s=90, edgecolor="black"
)
plt.scatter(
    scaler.inverse_transform(kmeans.cluster_centers_)[:, 0],
    scaler.inverse_transform(kmeans.cluster_centers_)[:, 1],
    s=250, c="black", marker="X", label="Centroids"
)
plt.title("Final Customer Segments (K-Means)")
plt.xlabel("Annual Income (k$)")
plt.ylabel("Spending Score (1-100)")
plt.legend(title="Cluster", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()


---
## Task 5 — Business Recommendations

*Replace the bracketed text below with your own observations once you've named the clusters in Task 4. Keep it specific and tied to a shopping mall in Tashkent — tenant mix, promotions, loyalty programs.*

**Segment 0 — [Name, e.g. "Budget-conscious visitors"]:**
[Who are they? What do they value? One concrete, local recommendation — e.g. "Offer targeted Friday evening discounts and a punch-card loyalty program at budget grocery and fast-fashion tenants to increase visit frequency without discounting margin on premium brands."]

**Segment 1 — [Name]:**
[...]

**Segment 2 — [Name]:**
[...]

**Segment 3 — [Name]:**
[...]

**Segment 4 — [Name]:**
[...]


---
## Submission checklist
- [ ] EDA: shape, dtypes, describe, histograms, pairplot
- [ ] K-Means: elbow + silhouette plots, justified K, labeled scatter plot
- [ ] Hierarchical: dendrogram, ward vs complete comparison
- [ ] Interpretation: named segments with `.groupby().mean()` table
- [ ] Business memo: 1 paragraph per segment, Tashkent-specific
